# 04 — Tokenizer Exploration

Explores the trained SLM tokenizer — vocabulary, special tokens, fertility, encoding examples, chat template formatting, and comparison with GPT-2.

Run after `python tokenizer/train_tokenizer.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import re
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from tokenizers import Tokenizer

DATA_DIR = Path('../data')
TOKENIZER_DIR = DATA_DIR / 'tokenizer'

from tokenizer.train_tokenizer import SPECIAL_TOKENS, BOS_ID, EOS_ID, PAD_ID

## 1. Load the tokenizer

In [ ]:
tokenizer = Tokenizer.from_file(str(TOKENIZER_DIR / 'slm_tokenizer.json'))
vocab = tokenizer.get_vocab()

print(f"Vocab size:      {len(vocab):,}")
print(f"Special tokens:  {len(SPECIAL_TOKENS)}")
print(f"Regular tokens:  {len(vocab) - len(SPECIAL_TOKENS):,}")

## 2. Special token verification

In [ ]:
print(f"{'ID':>4}  {'Token':<25}  {'Status'}")
print("-" * 45)
all_ok = True
for expected_id, token in enumerate(SPECIAL_TOKENS):
    actual_id = tokenizer.token_to_id(token)
    ok = actual_id == expected_id
    if not ok:
        all_ok = False
    status = "✓" if ok else f"✗ got {actual_id}"
    print(f"  {expected_id:>2}  {token:<25}  {status}")

print()
print("All special tokens correct:" , "✓ YES" if all_ok else "✗ NO")

## 3. Encoding examples

In [ ]:
examples = [
    ("English prose",    "The transformer architecture revolutionized natural language processing."),
    ("Python code",      "def factorial(n):\n    return 1 if n <= 1 else n * factorial(n - 1)"),
    ("Math",             "The integral of x^2 from 0 to 1 equals 1/3."),
    ("Mixed",            "Use torch.nn.functional.softmax(logits, dim=-1) to get probabilities."),
    ("Short",            "Hello."),
    ("Numbers",          "The year 2024 saw 1,234,567 tokens processed at 42.0 tokens/sec."),
]

print(f"{'Example':<15}  {'Tokens':>6}  {'Words':>6}  {'tok/word':>8}  {'First 10 tokens'}")
print("-" * 90)

for label, text in examples:
    enc = tokenizer.encode(text)
    words = len(text.split())
    tok_per_word = len(enc.ids) / max(words, 1)
    first_tokens = [tokenizer.id_to_token(tid) for tid in enc.ids[:10]]
    print(f"  {label:<13}  {len(enc.ids):>6}  {words:>6}  {tok_per_word:>8.2f}  {first_tokens}")

## 4. Fertility by source

In [ ]:
validated_path = DATA_DIR / 'validated' / 'train.jsonl'

fertility_by_source = {'wikipedia': [], 'code': [], 'common_crawl': []}

if validated_path.exists():
    with open(validated_path) as f:
        for i, line in enumerate(f):
            if i >= 3000:
                break
            record = json.loads(line)
            src = record.get('source', 'unknown')
            text = record.get('text', '')
            if src not in fertility_by_source or not text:
                continue
            words = len(text.split())
            tokens = len(tokenizer.encode(text).ids)
            if words > 10:
                fertility_by_source[src].append(tokens / words)

    fig, ax = plt.subplots(figsize=(9, 4))
    colors = {'wikipedia': '#4C72B0', 'code': '#55A868', 'common_crawl': '#DD8452'}

    for source, fertilities in fertility_by_source.items():
        if fertilities:
            ax.hist(fertilities, bins=40, alpha=0.65,
                    color=colors[source], label=f"{source} (μ={np.mean(fertilities):.2f})",
                    edgecolor='white')

    ax.axvline(1.3, color='black', linestyle='--', alpha=0.5, label='GPT-2 reference (1.3)')
    ax.set_xlabel('Tokens per word (fertility)')
    ax.set_ylabel('Count')
    ax.set_title('Tokenizer Fertility by Source', fontsize=12, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("Mean fertility by source:")
    for src, fertilities in fertility_by_source.items():
        if fertilities:
            print(f"  {src:<20} {np.mean(fertilities):.3f} tokens/word")
else:
    print("Validated data not found — run validation first")

## 5. Chat template formatting

In [ ]:
def format_chat(messages):
    parts = []
    for msg in messages:
        role, content = msg['role'], msg['content']
        if role == 'system':
            parts.append(f"<|system|>{content}<|endofturn|>")
        elif role == 'user':
            parts.append(f"<|user|>{content}<|endofturn|>")
        elif role == 'assistant':
            parts.append(f"<|assistant|>{content}<|endofturn|>")
    parts.append("<|assistant|>")  # prompt model to respond
    return "".join(parts)

messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Write a Python function to compute Fibonacci numbers."},
    {"role": "assistant", "content": "<|code|>def fib(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a<|endofcode|>"},
    {"role": "user", "content": "Can you add memoization?"},
]

chat_str = format_chat(messages)
encoded = tokenizer.encode(chat_str)

print("Chat string:")
print(chat_str)
print()
print(f"Total tokens: {len(encoded.ids)}")

# Highlight special tokens in the token sequence
special_positions = [
    (i, tokenizer.id_to_token(tid))
    for i, tid in enumerate(encoded.ids)
    if tokenizer.id_to_token(tid) in SPECIAL_TOKENS
]
print(f"\nSpecial token positions:")
for pos, token in special_positions:
    print(f"  [{pos:>4}] {token}")

## 6. Comparison with GPT-2 tokenizer

In [ ]:
try:
    from transformers import AutoTokenizer
    gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')

    test_texts = [
        "The transformer architecture has revolutionized natural language processing.",
        "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
        "Machine learning models are trained on large datasets to learn patterns.",
        "The quick brown fox jumps over the lazy dog.",
        "import torch; x = torch.randn(batch_size, seq_len, hidden_size)",
    ]

    rows = []
    for text in test_texts:
        slm_tokens = len(tokenizer.encode(text).ids)
        gpt2_tokens = len(gpt2_tokenizer.encode(text))
        rows.append({
            'Text': text[:60] + '...' if len(text) > 60 else text,
            'SLM tokens': slm_tokens,
            'GPT-2 tokens': gpt2_tokens,
            'Difference': slm_tokens - gpt2_tokens,
        })

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print()
    print(f"SLM avg tokens:   {df['SLM tokens'].mean():.1f}")
    print(f"GPT-2 avg tokens: {df['GPT-2 tokens'].mean():.1f}")

except Exception as e:
    print(f"Could not load GPT-2 tokenizer: {e}")

## 7. Vocabulary composition

In [ ]:
# Analyze token character composition
alpha_tokens = 0
numeric_tokens = 0
mixed_tokens = 0
punctuation_tokens = 0
byte_tokens = 0
special_count = len(SPECIAL_TOKENS)

for token, tid in vocab.items():
    if token in SPECIAL_TOKENS:
        continue
    # Decode byte-level token
    clean = token.replace('Ġ', ' ').replace('Ċ', '\n').strip()
    if not clean:
        byte_tokens += 1
    elif clean.isalpha():
        alpha_tokens += 1
    elif clean.isnumeric():
        numeric_tokens += 1
    elif clean.isalnum():
        mixed_tokens += 1
    elif all(not c.isalnum() for c in clean):
        punctuation_tokens += 1
    else:
        mixed_tokens += 1

categories = {
    'Alphabetic': alpha_tokens,
    'Mixed/Code': mixed_tokens,
    'Punctuation': punctuation_tokens,
    'Numeric': numeric_tokens,
    'Byte/Special': byte_tokens + special_count,
}

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
wedges, texts, autotexts = ax.pie(
    categories.values(), labels=categories.keys(),
    autopct='%1.1f%%', colors=colors, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title('Vocabulary Composition by Token Type', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

for cat, count in categories.items():
    print(f"  {cat:<20} {count:>6,}  ({100*count/len(vocab):.1f}%)")

## 8. Encode/decode roundtrip test

In [ ]:
test_cases = [
    "The quick brown fox jumps over the lazy dog.",
    "def hello():\n    print('Hello, world!')",
    "1 + 1 = 2",
    "café, naïve, résumé",  # accented characters
    "🚀 emoji support",
    "<|user|>Hello<|endofturn|><|assistant|>",  # special tokens
]

print(f"{'Text':<50}  {'Tokens':>6}  {'Roundtrip'}")
print("-" * 80)
for text in test_cases:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids, skip_special_tokens=False)
    match = text == decoded
    status = "✓" if match else "✗"
    print(f"  {text[:48]:<48}  {len(encoded.ids):>6}  {status}")
    if not match:
        print(f"    Expected: {repr(text)}")
        print(f"    Got:      {repr(decoded)}")